In [20]:
import time
import cv2
import numba
import numpy as np
import pandas as pd
from typing import List, Tuple
import matplotlib.pyplot as plt

# Entropy Function

In [2]:
def my_entropy(input_image):

    arr = input_image.flatten().astype(np.int64)

    if arr.min() != 1:
        arr = arr - arr.min() + 1

    p = np.zeros(arr.max(), dtype=np.float64)
    for v in arr:
        p[v - 1] += 1

    p = p / p.sum()
    p = p[p != 0]
    entropy = np.sum(-p * np.log2(p))

    return entropy

## PSNR Function

In [3]:
def peak_signal_noise_ratio(image1: np.ndarray, image2: np.ndarray):
    if image1.shape != image2.shape:
        print('Input images don’t have the same shape')
        return -1

    max_value = max(image1.max(), image2.max())
    max_value_square = max_value ** 2

    # mean-square-error
    img1_flat = image1.flatten()
    img2_flat = image2.flatten()

    error = img1_flat - img2_flat
    error_square = error ** 2
    error_square_sum = np.sum(error_square)
    mean_error_square_sum = error_square_sum / len(img1_flat)

    if mean_error_square_sum == 0:
        return np.inf
    return 10 * np.log10(max_value_square / (mean_error_square_sum))

# Question1: Median Edge Predictor (MED)

In [4]:
def med_predictor(input_image):
    input_image = input_image.astype(np.int16)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.int16)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image


def med_reconstructor(error_image):
    error_image = error_image.astype(np.int16)
    H, W = error_image.shape

    prediction = np.zeros((H + 1, W + 1), dtype=np.int16)
    reconstructed_image = np.zeros((H, W), dtype=np.uint8)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = np.uint8(prediction[i, j])

    return reconstructed_image

## MED Evaluation

In [5]:
test_path = ['Images/CLIC_2025_1.png', 'Images/CLIC_2025_2.png', 'Images/Kodak_01.png', 'Images/Kodak_23.png',
             'Images/Livingroom.tif', 'Images/Bridge.tif', 'Images/Baboon.tif', 'Images/Peppers.bmp',
             'Images/MRI_1.tif', 'Images/MRI_2.tif', 'Images/Retina.tif', 'Images/Cells.png']

evaluation_table_med = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    med_error = med_predictor(image)
    med_reconstruct = med_reconstructor(med_error)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_med.append({'Image Name': name,'Image Entropy': my_entropy(image),
                                 'Error Entropy(MED)': my_entropy(med_error), 'PSNR (initial-reconstruct)': peak_signal_noise_ratio(image, med_reconstruct),
                                 'Runtime': runtime})

evaluation_table_med = pd.DataFrame(evaluation_table_med)
evaluation_table_med

,Image Name,Image Entropy,Error Entropy(MED),PSNR (initial-reconstruct),Runtime
0,CLIC_2025_1,7.573678,4.024232,inf,39.219324
1,CLIC_2025_2,7.615759,5.889772,inf,32.812091
2,Kodak_01,7.161006,5.511179,inf,2.409108
3,Kodak_23,7.251587,3.829204,inf,1.859460
4,Livingroom,7.295174,4.839180,inf,1.193444
5,Bridge,7.683018,5.668946,inf,2.013834
6,Baboon,7.292549,5.233969,inf,1.162116
7,Peppers,7.571478,4.843694,inf,1.259580
8,MRI_1,6.428016,3.765981,inf,1.233865
9,MRI_2,6.619796,3.612304,inf,1.205327


# Question2: Third Order Linear Predictor (Optimum Mode)

## Calculate Coefficients

In [6]:
def compute_third_order_coef(img):

    img = img.astype(np.float64)

    a = img[1:, :-1].flatten()
    b = img[:-1, 1:].flatten()
    c = img[:-1, :-1].flatten()
    x = img[1:, 1:].flatten()

    A = np.sum(a*a)
    B = np.sum(b*b)
    C = np.sum(c*c)
    D = np.sum(a*b)
    E = np.sum(a*c)
    F = np.sum(b*c)

    Xa = np.sum(a*x)
    Xb = np.sum(b*x)
    Xc = np.sum(c*x)

    M = np.array([
        [A, D, E, 1],
        [D, B, F, 1],
        [E, F, C, 1],
        [1, 1, 1, 0]
    ], dtype=np.float64)

    rhs = np.array([Xa, Xb, Xc, 1], dtype=np.float64)
    sol = np.linalg.solve(M, rhs)

    alpha, beta = sol[0], sol[1]
    return alpha, beta

## Three Optimum Predictor

In [7]:
def three_optimum_predictor(input_image):
    input_image = input_image.astype(dtype=np.float64)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.float64)
    alpha, beta = compute_third_order_coef(input_image)
    gamma = 1.0 - (alpha + beta)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]
            x = (alpha*a) + (beta*b) + (gamma*c)

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image, alpha, beta


def three_optimum_reconstructor(error_image, alpha, beta):
    error_image = error_image.astype(np.float64)
    H, W = error_image.shape
    gamma = 1.0 - (alpha + beta)

    prediction = np.zeros((H + 1, W + 1), dtype=np.float64)
    reconstructed_image = np.zeros((H, W), dtype=np.float64)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            x = (alpha*a) + (beta*b) + (gamma*c)

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = prediction[i, j]

    return reconstructed_image

## Evaluation

In [8]:
evaluation_table_three_order = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    three_opt_error, alpha, beta = three_optimum_predictor(image)
    three_opt_reconstruct = three_optimum_reconstructor(three_opt_error, alpha, beta)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_three_order.append({'Image Name': name, 'Error Entropy(OPT-3)': my_entropy(three_opt_error),
                                         'PSNR (initial-reconstruct)': peak_signal_noise_ratio(image, three_opt_reconstruct),
                                         'Runtime(Encode-Decode)': runtime})

evaluation_table_three_order = pd.DataFrame(evaluation_table_three_order)
evaluation_table_three_order

,Image Name,Error Entropy(OPT-3),PSNR (initial-reconstruct),Runtime(Encode-Decode)
0,CLIC_2025_1,3.572794,381.918793,9.018007
1,CLIC_2025_2,5.739044,inf,9.442390
2,Kodak_01,5.511926,inf,1.284174
3,Kodak_23,3.543218,inf,1.198205
4,Livingroom,4.830203,310.046808,0.819520
5,Bridge,5.599623,378.240748,0.813972
6,Baboon,5.015353,inf,0.740210
7,Peppers,4.712849,376.289565,0.759527
8,MRI_1,3.204443,285.439082,0.832107
9,MRI_2,2.975721,307.270460,0.786133


# Question3: Designing Predictor

## Method 1: Partitioning & MED

In [41]:
def med_predict(a, b, c):
    if c >= max(a, b):
        return min(a, b)
    elif c <= min(a, b):
        return max(a, b)
    else:
        return a + b - c


def my_predictor(input_image):
    img = input_image.astype(np.float64)

    partitions = np.array_split(img, 16, axis=0)
    error_partitions = []

    for part in partitions:
        h, w = part.shape
        err = np.zeros((h, w), dtype=np.float64)

        padded = np.pad(part, ((1, 0), (1, 0)), mode='constant')

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = padded[i, j - 1]
                b = padded[i - 1, j]
                c = padded[i - 1, j - 1]

                pred = med_predict(a, b, c)
                err[i - 1, j - 1] = part[i - 1, j - 1] - pred

        error_partitions.append(err)

    error_image = np.vstack(error_partitions)
    return error_image

def my_reconstructor(error_image):
    err = error_image.astype(np.float64)
    err_parts = np.array_split(err, 16, axis=0)

    reconstructed_parts = []

    for err_part in err_parts:
        h, w = err_part.shape

        pred_img = np.zeros((h + 1, w + 1), dtype=np.float64)
        rec = np.zeros((h, w), dtype=np.float64)

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = pred_img[i, j - 1]
                b = pred_img[i - 1, j]
                c = pred_img[i - 1, j - 1]

                pred = med_predict(a, b, c)

                val = pred + err_part[i - 1, j - 1]
                pred_img[i, j] = val
                rec[i - 1, j - 1] = val

        reconstructed_parts.append(rec)

    reconstructed_image = np.vstack(reconstructed_parts)
    return reconstructed_image

## Evaluation of Method 1

In [44]:
evaluation_table_one = []

for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    my_predictor_error = my_predictor(image)
    my_predictor_reconstruct = my_reconstructor(my_predictor_error)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_one.append({'Image Name': name, 'Error Entropy(MED-with-Partitioning)': my_entropy(my_predictor_error),
                                 'PSNR': peak_signal_noise_ratio(image, my_predictor_reconstruct),
                                 'Runtime': runtime})

evaluation_table_one = pd.DataFrame(evaluation_table_one)

# Average Row
error_entropy_mean_one = evaluation_table_one['Error Entropy(MED-with-Partitioning)'].mean()
runtime_mean_one = evaluation_table_one['Runtime'].mean()
evaluation_table_one.loc[len(evaluation_table_one)] = {"Image Name": "Average", 'Error Entropy(MED-with-Partitioning)': error_entropy_mean_one,
                                                       "PSNR": np.inf, "Runtime": runtime_mean_one}

evaluation_table_one

,Image Name,Error Entropy(MED-with-Partitioning),PSNR,Runtime
0,CLIC_2025_1,4.026119,inf,9.810825
1,CLIC_2025_2,5.892087,inf,10.196966
2,Kodak_01,5.523678,inf,1.348761
3,Kodak_23,3.840914,inf,1.492096
4,Livingroom,4.861838,inf,1.014786
5,Bridge,5.681421,inf,0.968485
6,Baboon,5.254854,inf,0.981955
7,Peppers,4.850672,inf,0.947756
8,MRI_1,3.794911,inf,0.842732
9,MRI_2,3.642836,inf,0.908318


## Method2: Second Order Optimum LS Predictor with Partitioning

In [16]:
def compute_coefficients(part):

    part = part.astype(np.float64)

    a = part[1:, :-1].ravel()
    b = part[:-1, 1:].ravel()
    x = part[1:, 1:].ravel()

    diff = (b - a)
    denom = np.sum(diff * diff)
    if denom == 0:
        return 0.0
    numer = np.sum((x - b) * diff)
    rho = numer / denom
    return float(rho)


def my_predictor_second(input_image, n_partitions=4, axis='rows'):

    img = input_image.astype(np.float64)

    if axis == 'rows':
        parts = np.array_split(img, n_partitions, axis=0)
    elif axis == 'cols':
        parts = np.array_split(img, n_partitions, axis=1)

    error_parts = []
    coef_list = []

    for part in parts:
        h, w = part.shape
        coef = compute_coefficients(part)
        coef_list.append(coef)

        padded = np.pad(part, ((1, 0), (1, 0)), mode='constant', constant_values=0.0)
        err = np.zeros((h, w), dtype=np.float64)

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = padded[i, j - 1]
                b = padded[i - 1, j]
                pred = b + coef * (b - a)
                err[i - 1, j - 1] = part[i - 1, j - 1] - pred

        error_parts.append(err)

    if axis == 'rows':
        error_image = np.vstack(error_parts)
    else:
        error_image = np.hstack(error_parts)

    return error_image, coef_list


def my_reconstructor_second(error_image, rhos, axis='rows'):
    err = error_image
    if axis == 'rows':
        err_parts = np.array_split(err, len(rhos), axis=0)
    else:
        err_parts = np.array_split(err, len(rhos), axis=1)

    recon_parts = []

    for err_part, rho in zip(err_parts, rhos):
        h, w = err_part.shape
        pred_buf = np.zeros((h + 1, w + 1), dtype=np.float64)
        rec = np.zeros((h, w), dtype=np.float64)

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = pred_buf[i, j - 1]
                b = pred_buf[i - 1, j]
                pred = b + rho * (b - a)
                val = pred + err_part[i - 1, j - 1]
                pred_buf[i, j] = val
                rec[i - 1, j - 1] = val

        recon_parts.append(rec)

    if axis == 'rows':
        reconstructed = np.vstack(recon_parts)
    else:
        reconstructed = np.hstack(recon_parts)

    return reconstructed

## Evaluation

## Running with 4 partitions

In [17]:
evaluation_table_my_predictor_second = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    my_predictor_error_second, coef = my_predictor_second(image, n_partitions=4)
    my_predictor_reconstruct_second = my_reconstructor_second(my_predictor_error_second, coef)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_my_predictor_second.append({'Image Name': name, 'Error Entropy(my_method_second)': my_entropy(my_predictor_error_second),
                                                 'PSNR (initial-reconstruct)': peak_signal_noise_ratio(image, my_predictor_reconstruct_second),
                                                 'Runtime(Encode-Decode)': runtime})

evaluation_table_my_predictor_second = pd.DataFrame(evaluation_table_my_predictor_second)
evaluation_table_my_predictor_second

,Image Name,Error Entropy(my_method_second),PSNR (initial-reconstruct),Runtime(Encode-Decode)
0,CLIC_2025_1,3.700764,inf,6.016962
1,CLIC_2025_2,5.816804,inf,6.392510
2,Kodak_01,5.678268,inf,0.871006
3,Kodak_23,3.665550,inf,0.916275
4,Livingroom,5.024417,inf,0.650882
5,Bridge,5.704642,inf,0.632287
6,Baboon,5.398928,inf,0.608624
7,Peppers,4.675139,inf,0.612689
8,MRI_1,4.004428,inf,0.685678
9,MRI_2,3.669168,inf,0.566364


## Running with 2 partitions

In [18]:
evaluation_table_my_predictor_second = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    my_predictor_error_second, rhos = my_predictor_second(image, n_partitions=2, axis='cols')
    my_predictor_reconstruct_second = my_reconstructor_second(my_predictor_error_second, rhos, axis='cols')
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_my_predictor_second.append({'Image Name': name, 'Error Entropy(my_method_second)': my_entropy(my_predictor_error_second),
                                                 'PSNR (initial-reconstruct)': peak_signal_noise_ratio(image, my_predictor_reconstruct_second),
                                                 'Runtime(Encode-Decode)': runtime})

evaluation_table_my_predictor_second = pd.DataFrame(evaluation_table_my_predictor_second)
evaluation_table_my_predictor_second

,Image Name,Error Entropy(my_method_second),PSNR (initial-reconstruct),Runtime(Encode-Decode)
0,CLIC_2025_1,3.683566,inf,5.456336
1,CLIC_2025_2,5.829822,inf,6.303531
2,Kodak_01,5.662969,inf,0.834635
3,Kodak_23,3.630878,inf,0.856704
4,Livingroom,5.087795,inf,0.521418
5,Bridge,5.716975,inf,0.590457
6,Baboon,5.389363,inf,0.508181
7,Peppers,4.653548,inf,0.553130
8,MRI_1,3.997783,inf,0.539492
9,MRI_2,3.688237,inf,0.544497


In [19]:
print(evaluation_table_med['Error Entropy(MED)'].mean())
print(evaluation_table_three_order['Error Entropy(OPT-3)'].mean())
print(evaluation_table_my_predictor['Error Entropy(my_method)'].mean())
print(evaluation_table_my_predictor_second['Error Entropy(my_method_second)'].mean())

4.493368715834052
4.217094265647606
4.507836740963415
4.45949909965319
